# 위험 소리 분류 파인튜닝 Colab

이 노트북은 회의록/중간보고서에 적힌 후보 모델과 데이터셋을 기준으로, 청각장애인 지원 키링/앱 서비스에서 중요한 **위험·주의 소리**를 우선 분류하도록 구성했습니다.

- 기본 타깃 클래스: `car_horn`, `siren`, `glass_breaking`, `explosion_or_gunshot`, `construction_or_machine`, `fire_alarm`, `baby_cry`, `doorbell_or_bell`, `other`
- 기본 자동 다운로드: ESC-50
- 선택 다운로드/로딩: UrbanSound8K, FSD50K, AudioSet, AI Hub 소음 환경 음성인식 데이터
- 위험 소리 성능 보강: 클래스 가중치, `other` 샘플 제한, augmentation, validation F1 확인

> Colab에서 먼저 `런타임 > 런타임 유형 변경 > GPU`를 선택하세요. 빠른 실험은 `EPOCHS=3`, `MAX_SAMPLES_PER_CLASS=300` 정도로 시작하고, 최종 비교 때 늘리는 방식이 안전합니다.

## 모델: YAMNet
YAMNet은 TF Hub의 AudioSet 사전학습 모델을 feature extractor로 쓰고, 각 오디오 window embedding 위에 위험 소리 classifier head를 학습합니다.

In [1]:
# Colab 설치 셀: YAMNet / TensorFlow Hub
!pip -q install tensorflow tensorflow-hub librosa soundfile pandas scikit-learn tqdm matplotlib kaggle

In [2]:
import os
import re
import json
import math
import random
import zipfile
import shutil
import subprocess
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================================================
# 1) 데이터셋 설정
# =========================================================
DATA_ROOT = Path('/content/sound_data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 회의록 사진/보고서에 있는 데이터셋을 모두 지원합니다.
# ESC-50은 자동 다운로드, UrbanSound8K는 Kaggle 계정이 있으면 자동 다운로드,
# FSD50K/AudioSet/AI Hub는 크기·라이선스·로그인 이슈 때문에 Google Drive에 받아둔 경로를 연결하는 방식입니다.
DATASETS_TO_USE = [
    'esc50',
    'urbansound8k',
    'fsd50k',
    'audioset',
    'aihub_noise',
]

USE_KAGGLE_URBANSOUND8K = False  # Kaggle API 토큰(kaggle.json)을 넣었다면 True로 바꾸세요.

# 큰 데이터셋은 Drive에 직접 다운로드한 뒤 아래 경로를 맞추세요.
FSD50K_ROOT = Path('/content/drive/MyDrive/datasets/FSD50K')
AUDIOSET_ROOT = Path('/content/drive/MyDrive/datasets/AudioSet_prepared')
AIHUB_NOISE_ROOT = Path('/content/drive/MyDrive/datasets/AIHub_noise_speech')

# Colab 빠른 실험용 제한. 최종 실험 때 늘리거나 None으로 바꾸세요.
MAX_SAMPLES_PER_CLASS = 600
MAX_OTHER_SAMPLES = 600
MIN_SAMPLES_PER_CLASS = 5

# 위험/주의 분류 타깃. 실제 서비스에서는 필요에 따라 baby_cry/doorbell을 별도 "주의" 레벨로 매핑하면 됩니다.
TARGET_LABELS = [
    'car_horn',
    'siren',
    'glass_breaking',
    'explosion_or_gunshot',
    'construction_or_machine',
    'fire_alarm',
    'baby_cry',
    'doorbell_or_bell',
    'other',
]
DANGER_LABELS = {
    'car_horn', 'siren', 'glass_breaking', 'explosion_or_gunshot',
    'construction_or_machine', 'fire_alarm'
}

# 라벨 매핑 우선순위: 여러 라벨이 동시에 있을 때 위험도가 높은 쪽을 선택합니다.
LABEL_PRIORITY = [
    'explosion_or_gunshot',
    'glass_breaking',
    'fire_alarm',
    'siren',
    'car_horn',
    'construction_or_machine',
    'baby_cry',
    'doorbell_or_bell',
    'other',
]

KEYWORD_RULES = [
    ('explosion_or_gunshot', [
        'explosion', 'explode', 'blast', 'gun_shot', 'gunshot', 'gun shot', 'gunfire', 'fireworks', 'firework',
        '폭발', '폭발음', '총성', '총소리', '사격', '폭죽'
    ]),
    ('glass_breaking', [
        'glass_breaking', 'glass breaking', 'glass_break', 'breaking_glass', 'shatter', 'glass',
        '유리', '유리깨짐', '유리 깨', '파손'
    ]),
    ('fire_alarm', [
        'fire_alarm', 'smoke_alarm', 'smoke detector', 'alarm', 'beep', 'buzzer',
        '화재경보', '화재 경보', '경보기', '경보음', '경보'
    ]),
    ('siren', [
        'siren', 'civil_defense_siren', 'emergency_vehicle', 'ambulance', 'police_siren',
        '싸이렌', '사이렌', '구급차', '경찰차'
    ]),
    ('car_horn', [
        'car_horn', 'vehicle_horn', 'air_horn', 'horn', 'honk',
        '경적', '경적소음', '클락션', '자동차 경적'
    ]),
    ('construction_or_machine', [
        'jackhammer', 'drilling', 'drill', 'chainsaw', 'hand_saw', 'saw', 'hammer', 'construction', 'machine',
        '착암', '드릴', '전동해머', '해머드릴', '기계톱', '절단기', '공사', '공장', '프레스'
    ]),
    ('baby_cry', [
        'crying_baby', 'baby_cry', 'baby crying', 'infant cry',
        '아기 울음', '아이 울음', '영아', '울음소리'
    ]),
    ('doorbell_or_bell', [
        'doorbell', 'door_bell', 'bell', 'door_knock', 'knock', 'church_bells',
        '초인종', '벨소리', '노크', '문 두드림', '방문'
    ]),
]

LABEL2ID = {label: i for i, label in enumerate(TARGET_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

def normalize_text(x):
    x = str(x).strip().lower()
    x = x.replace('-', '_').replace('/', '_').replace('\\', '_')
    x = re.sub(r'\s+', ' ', x)
    return x

def canonicalize_label(raw_label):
    """원본 데이터셋 라벨을 프로젝트 타깃 라벨로 변환합니다."""
    s = normalize_text(raw_label)
    for canonical, keys in KEYWORD_RULES:
        for key in keys:
            if normalize_text(key) in s:
                return canonical
    return 'other'

def pick_best_label(raw_labels):
    mapped = [canonicalize_label(x) for x in raw_labels if str(x).strip()]
    if not mapped:
        return 'other'
    for label in LABEL_PRIORITY:
        if label in mapped:
            return label
    return 'other'

def maybe_download_esc50(root=DATA_ROOT):
    target = root / 'ESC-50-master'
    if target.exists():
        return target
    url = 'https://github.com/karolpiczak/ESC-50/archive/master.zip'
    zip_path = root / 'ESC-50-master.zip'
    print(f'[ESC-50] downloading: {url}')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(root)
    zip_path.unlink(missing_ok=True)
    return target

def maybe_download_urbansound8k(root=DATA_ROOT):
    target = root / 'UrbanSound8K'
    if target.exists():
        return target
    if not USE_KAGGLE_URBANSOUND8K:
        print('[UrbanSound8K] USE_KAGGLE_URBANSOUND8K=False라서 자동 다운로드를 건너뜁니다.')
        return target
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    if not kaggle_json.exists():
        print('[UrbanSound8K] Kaggle 토큰이 없습니다. Colab에 kaggle.json을 업로드한 뒤 ~/.kaggle/kaggle.json에 배치하세요.')
        return target
    target.mkdir(parents=True, exist_ok=True)
    print('[UrbanSound8K] downloading from Kaggle: chrisfilo/urbansound8k')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'chrisfilo/urbansound8k', '-p', str(target)], check=True)
    for zp in target.glob('*.zip'):
        with zipfile.ZipFile(zp, 'r') as zf:
            zf.extractall(target)
        zp.unlink(missing_ok=True)
    return target

def load_esc50(root):
    csv_candidates = list(Path(root).rglob('esc50.csv'))
    if not csv_candidates:
        return []
    meta_path = csv_candidates[0]
    base = meta_path.parent.parent
    audio_dir = base / 'audio'
    meta = pd.read_csv(meta_path)
    samples = []
    for _, row in meta.iterrows():
        raw = str(row.get('category', ''))
        label = canonicalize_label(raw)
        path = audio_dir / str(row['filename'])
        if path.exists():
            samples.append({'path': str(path), 'label': label, 'source': 'ESC-50', 'raw_label': raw})
    return samples

def load_urbansound8k(root):
    root = Path(root)
    csv_candidates = list(root.rglob('UrbanSound8K.csv'))
    if not csv_candidates:
        return []
    meta_path = csv_candidates[0]
    # 보통 .../UrbanSound8K/metadata/UrbanSound8K.csv 구조입니다.
    base = meta_path.parent.parent if meta_path.parent.name.lower() == 'metadata' else meta_path.parent
    meta = pd.read_csv(meta_path)
    samples = []
    for _, row in meta.iterrows():
        raw = str(row.get('class', row.get('classID', '')))
        label = canonicalize_label(raw)
        fname = str(row['slice_file_name'])
        fold = f"fold{int(row['fold'])}" if 'fold' in row else ''
        candidates = [base / 'audio' / fold / fname, base / fold / fname]
        path = next((p for p in candidates if p.exists()), None)
        if path is None:
            matches = list(base.rglob(fname))
            path = matches[0] if matches else None
        if path and path.exists():
            samples.append({'path': str(path), 'label': label, 'source': 'UrbanSound8K', 'raw_label': raw})
    return samples

def load_fsd50k(root):
    root = Path(root)
    if not root.exists():
        return []
    csvs = []
    for name in ['dev.csv', 'eval.csv']:
        csvs.extend(root.rglob(name))
    samples = []
    for csv_path in csvs:
        try:
            meta = pd.read_csv(csv_path)
        except Exception as e:
            print(f'[FSD50K] CSV read fail: {csv_path} / {e}')
            continue
        split = 'dev' if 'dev' in csv_path.name.lower() else 'eval'
        audio_dirs = list(root.rglob(f'FSD50K.{split}_audio')) + list(root.rglob(f'{split}_audio'))
        audio_dir = audio_dirs[0] if audio_dirs else root
        fname_col = 'fname' if 'fname' in meta.columns else meta.columns[0]
        labels_col = 'labels' if 'labels' in meta.columns else None
        if labels_col is None:
            continue
        for _, row in meta.iterrows():
            fname = str(row[fname_col])
            raw_labels = str(row[labels_col]).split(',')
            label = pick_best_label(raw_labels)
            candidates = [audio_dir / f'{fname}.wav', audio_dir / fname]
            path = next((p for p in candidates if p.exists()), None)
            if path is None:
                matches = list(root.rglob(f'{fname}.wav'))
                path = matches[0] if matches else None
            if path and path.exists():
                samples.append({'path': str(path), 'label': label, 'source': 'FSD50K', 'raw_label': ','.join(raw_labels)})
    return samples

def load_audiofolder_by_parent(root, source_name):
    """root 아래 wav/flac/mp3 파일을 찾고, 상위 폴더명을 라벨로 사용합니다. AudioSet을 직접 wav로 정리한 경우 유용합니다."""
    root = Path(root)
    if not root.exists():
        return []
    exts = ['*.wav', '*.flac', '*.mp3', '*.ogg', '*.m4a']
    files = []
    for ext in exts:
        files.extend(root.rglob(ext))
    samples = []
    for path in files:
        raw = path.parent.name
        label = canonicalize_label(raw)
        samples.append({'path': str(path), 'label': label, 'source': source_name, 'raw_label': raw})
    return samples

def flatten_values(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield str(k)
            yield from flatten_values(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from flatten_values(v)
    else:
        yield str(obj)

def load_aihub_noise(root):
    """AI Hub JSON 구조가 세부 버전마다 달라서, JSON 전체 텍스트에서 소음 라벨 키워드를 탐색합니다."""
    root = Path(root)
    if not root.exists():
        return []
    samples = []
    json_files = list(root.rglob('*.json'))
    used_audio = set()
    for jp in tqdm(json_files, desc='AI Hub JSON scan'):
        try:
            data = json.loads(jp.read_text(encoding='utf-8'))
        except Exception:
            try:
                data = json.loads(jp.read_text(encoding='cp949'))
            except Exception:
                continue
        raw_text = ' '.join(flatten_values(data))
        label = canonicalize_label(raw_text)
        # 같은 stem 또는 같은 폴더의 오디오 파일을 찾습니다.
        candidates = []
        for ext in ['.wav', '.flac', '.mp3', '.m4a']:
            candidates.append(jp.with_suffix(ext))
        if not any(p.exists() for p in candidates):
            candidates.extend(list(jp.parent.glob('*.wav')))
            candidates.extend(list(jp.parent.glob('*.flac')))
        path = next((p for p in candidates if p.exists() and str(p) not in used_audio), None)
        if path is not None:
            used_audio.add(str(path))
            samples.append({'path': str(path), 'label': label, 'source': 'AIHub_noise', 'raw_label': raw_text[:160]})
    # JSON이 없다면 폴더명 기반으로라도 로딩합니다.
    if not samples:
        samples = load_audiofolder_by_parent(root, 'AIHub_noise')
    return samples

def limit_samples(df):
    pieces = []
    for label, g in df.groupby('label'):
        cap = MAX_OTHER_SAMPLES if label == 'other' else MAX_SAMPLES_PER_CLASS
        if cap is not None and len(g) > cap:
            g = g.sample(cap, random_state=SEED)
        pieces.append(g)
    return pd.concat(pieces, ignore_index=True) if pieces else df

def safe_train_test_split(df, test_size, stratify_col='label'):
    counts = df[stratify_col].value_counts()
    stratify = df[stratify_col] if len(counts) > 1 and counts.min() >= 2 else None
    return train_test_split(df, test_size=test_size, random_state=SEED, stratify=stratify)

def build_metadata():
    all_samples = []

    if 'esc50' in DATASETS_TO_USE:
        esc_root = maybe_download_esc50(DATA_ROOT)
        s = load_esc50(esc_root)
        print(f'[ESC-50] {len(s)} samples')
        all_samples.extend(s)

    if 'urbansound8k' in DATASETS_TO_USE:
        urban_root = maybe_download_urbansound8k(DATA_ROOT)
        s = load_urbansound8k(urban_root)
        print(f'[UrbanSound8K] {len(s)} samples')
        all_samples.extend(s)

    if 'fsd50k' in DATASETS_TO_USE:
        s = load_fsd50k(FSD50K_ROOT)
        print(f'[FSD50K] {len(s)} samples from {FSD50K_ROOT}')
        all_samples.extend(s)

    if 'audioset' in DATASETS_TO_USE:
        # AudioSet은 공식적으로 YouTube segment/ontology 중심이라, wav로 준비한 폴더를 읽는 형태로 제공합니다.
        s = load_audiofolder_by_parent(AUDIOSET_ROOT, 'AudioSet_prepared')
        print(f'[AudioSet prepared audiofolder] {len(s)} samples from {AUDIOSET_ROOT}')
        all_samples.extend(s)

    if 'aihub_noise' in DATASETS_TO_USE:
        s = load_aihub_noise(AIHUB_NOISE_ROOT)
        print(f'[AI Hub noise] {len(s)} samples from {AIHUB_NOISE_ROOT}')
        all_samples.extend(s)

    df = pd.DataFrame(all_samples)
    if df.empty:
        raise RuntimeError('로드된 오디오 샘플이 없습니다. ESC-50 다운로드 상태 또는 Drive/Kaggle 경로를 확인하세요.')

    df = df.drop_duplicates('path').reset_index(drop=True)
    df = df[df['label'].isin(TARGET_LABELS)].reset_index(drop=True)
    df = limit_samples(df)

    counts = df['label'].value_counts()
    keep = counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
    dropped = sorted(set(df['label']) - set(keep))
    if dropped:
        print('[WARN] 샘플 수가 너무 적어 제외되는 클래스:', dropped)
        df = df[df['label'].isin(keep)].reset_index(drop=True)

    active_labels = [x for x in TARGET_LABELS if x in set(df['label'])]
    label2id = {label: i for i, label in enumerate(active_labels)}
    id2label = {i: label for label, i in label2id.items()}
    df['label_id'] = df['label'].map(label2id).astype(int)

    train_val_df, test_df = safe_train_test_split(df, test_size=0.15)
    train_df, val_df = safe_train_test_split(train_val_df, test_size=0.1765)  # 약 70/15/15

    print('\n[전체 라벨 분포]')
    print(df.groupby(['label', 'source']).size().unstack(fill_value=0))
    print('\n[split sizes]', {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
    print('[active labels]', active_labels)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True), active_labels, label2id, id2label

def load_audio_np(path, sr=16000, clip_seconds=5.0, random_crop=False):
    """오디오를 mono float32 [-1, 1] 근사 범위로 읽고 고정 길이로 crop/pad합니다."""
    y, _ = librosa.load(path, sr=sr, mono=True)
    if not np.isfinite(y).all():
        y = np.nan_to_num(y)
    target_len = int(sr * clip_seconds)
    if len(y) == 0:
        y = np.zeros(target_len, dtype=np.float32)
    if len(y) > target_len:
        if random_crop:
            start = random.randint(0, len(y) - target_len)
        else:
            start = max(0, (len(y) - target_len) // 2)
        y = y[start:start + target_len]
    elif len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    y = y.astype(np.float32)
    peak = np.max(np.abs(y)) if len(y) else 0
    if peak > 1.0:
        y = y / peak
    return y

def augment_waveform_np(y, sr):
    y = y.copy()
    if random.random() < 0.50:
        gain_db = random.uniform(-6.0, 6.0)
        y *= 10 ** (gain_db / 20.0)
    if random.random() < 0.35:
        shift = random.randint(-sr // 5, sr // 5)
        y = np.roll(y, shift)
    if random.random() < 0.35:
        noise_scale = random.uniform(0.0005, 0.006)
        y = y + np.random.randn(len(y)).astype(np.float32) * noise_scale
    return np.clip(y, -1.0, 1.0).astype(np.float32)

def make_class_weights(train_df, active_labels):
    labels = np.array([active_labels[i] for i in train_df['label_id'].values])
    classes = np.array(active_labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    return {i: float(w) for i, w in enumerate(weights)}

def save_labels_json(path, id2label):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({int(k): v for k, v in id2label.items()}, ensure_ascii=False, indent=2), encoding='utf-8')
    print('saved:', path)

## 데이터셋 로딩
ESC-50은 자동 다운로드됩니다. UrbanSound8K/FSD50K/AudioSet/AI Hub는 설정한 경로가 있으면 함께 합쳐집니다.

In [3]:
train_df, val_df, test_df, active_labels, label2id, id2label = build_metadata()
class_weights = make_class_weights(train_df, active_labels)
print('class_weights:', class_weights)

[ESC-50] downloading: https://github.com/karolpiczak/ESC-50/archive/master.zip
[ESC-50] 2000 samples
[UrbanSound8K] USE_KAGGLE_URBANSOUND8K=False라서 자동 다운로드를 건너뜁니다.
[UrbanSound8K] 0 samples
[FSD50K] 0 samples from /content/drive/MyDrive/datasets/FSD50K
[AudioSet prepared audiofolder] 0 samples from /content/drive/MyDrive/datasets/AudioSet_prepared
[AI Hub noise] 0 samples from /content/drive/MyDrive/datasets/AIHub_noise_speech

[전체 라벨 분포]
source                   ESC-50
label                          
baby_cry                     40
car_horn                     40
construction_or_machine     120
doorbell_or_bell             80
explosion_or_gunshot         40
fire_alarm                   40
glass_breaking               40
other                       600
siren                        40

[split sizes] {'train': 727, 'val': 157, 'test': 156}
[active labels] ['car_horn', 'siren', 'glass_breaking', 'explosion_or_gunshot', 'construction_or_machine', 'fire_alarm', 'baby_cry', 'doorbell_or_bell'

## YAMNet embedding 추출 및 classifier head 학습

In [4]:
import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt

SAMPLE_RATE = 16000
CLIP_SECONDS = 5.0
EPOCHS = 8
BATCH_SIZE = 64

print('TensorFlow:', tf.__version__)
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

def extract_embeddings(df, split_name, use_all_windows=True):
    X, y, paths = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'YAMNet embeddings: {split_name}'):
        wav = load_audio_np(row['path'], sr=SAMPLE_RATE, clip_seconds=CLIP_SECONDS, random_crop=(split_name == 'train'))
        scores, embeddings, spectrogram = yamnet(wav)
        emb = embeddings.numpy().astype(np.float32)
        if use_all_windows:
            X.extend([e for e in emb])
            y.extend([int(row['label_id'])] * len(emb))
            paths.extend([row['path']] * len(emb))
        else:
            X.append(emb.mean(axis=0))
            y.append(int(row['label_id']))
            paths.append(row['path'])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int64), paths

X_train, y_train, _ = extract_embeddings(train_df, 'train', use_all_windows=True)
X_val, y_val, _ = extract_embeddings(val_df, 'val', use_all_windows=True)
X_test, y_test, _ = extract_embeddings(test_df, 'test', use_all_windows=True)
print(X_train.shape, y_train.shape)

num_classes = len(active_labels)
inputs = tf.keras.Input(shape=(1024,), name='yamnet_embedding')
x = tf.keras.layers.BatchNormalization()(inputs)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.35)(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='danger_sound')(x)
classifier = tf.keras.Model(inputs, outputs)

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
classifier.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('/content/yamnet_danger_classifier.keras', monitor='val_accuracy', save_best_only=True)
]

history = classifier.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

TensorFlow: 2.20.0


YAMNet embeddings: train:   0%|          | 0/727 [00:00<?, ?it/s]

YAMNet embeddings: val:   0%|          | 0/157 [00:00<?, ?it/s]

YAMNet embeddings: test:   0%|          | 0/156 [00:00<?, ?it/s]

(7270, 1024) (7270,)


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ yamnet_embedding (InputLayer)   │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ danger_sound (Dense)            │ (None, 9)              │         1,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 300,553 (1.15 MB)

 Trainable params: 298,505 (1.14 MB)

 Non-trainable params: 2,048 (8.00 KB)

Epoch 1/8
114/114 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.5550 - loss: 1.1514 - val_accuracy: 0.6732 - val_loss: 0.9097
Epoch 2/8
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6612 - loss: 0.6734 - val_accuracy: 0.6892 - val_loss: 0.8158
Epoch 3/8
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6926 - loss: 0.5891 - val_accuracy: 0.7083 - val_loss: 0.7874
Epoch 4/8
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7001 - loss: 0.5915 - val_accuracy: 0.6764 - val_loss: 0.8639
Epoch 5/8
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7121 - loss: 0.5240 - val_accuracy: 0.7032 - val_loss: 0.7586
Epoch 6/8
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7264 - loss: 0.4993 - val_accuracy: 0.7083 - val_loss: 0.7619
Epoch 7/8
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7294 - loss: 0.4983 - val_accuracy: 0.6898 - val_loss: 0.7874


## 평가

In [5]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

y_prob = classifier.predict(X_test, batch_size=BATCH_SIZE)
y_pred = y_prob.argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=active_labels, zero_division=0))
print('accuracy:', accuracy_score(y_test, y_pred))
print('macro_f1:', f1_score(y_test, y_pred, average='macro', zero_division=0))
print('confusion_matrix:\n', confusion_matrix(y_test, y_pred))

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
                         precision    recall  f1-score   support

               car_horn       0.65      0.78      0.71        60
                  siren       0.83      0.83      0.83        60
         glass_breaking       0.20      0.95      0.33        60
   explosion_or_gunshot       0.62      0.80      0.70        60
construction_or_machine       0.58      0.90      0.70       180
             fire_alarm       0.64      0.95      0.77        60
               baby_cry       0.57      0.97      0.72        60
       doorbell_or_bell       0.67      0.54      0.60       120
                  other       0.96      0.53      0.69       900

               accuracy                           0.66      1560
              macro avg       0.64      0.81      0.67      1560
           weighted avg       0.81      0.66      0.68      1560

accuracy: 0.6564102564102564
macro_f1: 0.6724797599627776
confusion_matrix:
 [[ 47   0   6   0   5   0   0   0  

## 파일 하나 예측

In [6]:
def predict_file_yamnet(audio_path, top_k=5):
    wav = load_audio_np(audio_path, sr=SAMPLE_RATE, clip_seconds=CLIP_SECONDS, random_crop=False)
    _, embeddings, _ = yamnet(wav)
    probs = classifier.predict(embeddings.numpy(), verbose=0).mean(axis=0)
    order = probs.argsort()[::-1]
    return [(active_labels[i], float(probs[i])) for i in order[:top_k]]

# 예시: test_df 첫 파일
sample_path = test_df.iloc[0]['path']
print(sample_path, test_df.iloc[0]['label'])
print(predict_file_yamnet(sample_path))

/content/sound_data/ESC-50-master/audio/4-204115-A-39.wav glass_breaking
[('glass_breaking', 0.3704085946083069), ('doorbell_or_bell', 0.16520752012729645), ('construction_or_machine', 0.12792155146598816), ('car_horn', 0.10299865901470184), ('other', 0.0956302061676979)]


## 저장 / TFLite 변환
YAMNet 본체는 TF Hub 모델을 그대로 쓰고, 여기서는 classifier head를 저장합니다. 온디바이스에서는 YAMNet embedding 추출부 + 이 head를 연결하거나, TFLite audio classifier 형태로 별도 통합하세요.

In [7]:
import json

classifier.save('/content/yamnet_danger_classifier.keras')
save_labels_json('/content/yamnet_labels.json', id2label)

converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
Path('/content/yamnet_danger_classifier_head_int8_dynamic.tflite').write_bytes(tflite_model)
print('saved /content/yamnet_danger_classifier_head_int8_dynamic.tflite')

saved: /content/yamnet_labels.json
Saved artifact at '/tmp/tmpodeaqys2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1024), dtype=tf.float32, name='yamnet_embedding')
Output Type:
  TensorSpec(shape=(None, 9), dtype=tf.float32, name=None)
Captures:
  138755400770832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138755400764304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138755400771024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138755400765648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138755400764688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138755400773904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138755400776016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138755400763536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138755400775440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138755407049936: TensorSpec(shape=(), dtype=tf.resou